In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# 【读取步骤】
# 读取 credit_card_balance 数据
# ============================================================


# 1. 读取原始信用卡月度数据
credit_card = pd.read_csv(
    "data/raw/home_credit/credit_card_balance.csv"
)


# 2. 查看数据规模
print(
    "credit_card_balance shape:",
    credit_card.shape
)


# 3. 查看前几行
display(
    credit_card.head()
)


# 4. 查看字段类型
credit_card.info()

In [ ]:
# 1. 查看总月度记录数量
print(
    "总月度记录数量:",
    credit_card.shape[0]
)


# 2. 查看唯一客户数量
print(
    "唯一客户数量:",
    credit_card["SK_ID_CURR"].nunique()
)


# 3. 查看唯一历史信用卡账户数量
print(
    "唯一信用卡账户数量:",
    credit_card["SK_ID_PREV"].nunique()
)

In [ ]:
# 1. 检查完全重复行
print(
    "完全重复行数量:",
    credit_card.duplicated().sum()
)


# 2. 检查同一信用卡同一月份是否重复
print(
    "账户月份组合重复数量:",
    credit_card.duplicated(
        subset=[
            "SK_ID_PREV",
            "MONTHS_BALANCE"
        ]
    ).sum()
)

In [ ]:
# 1. 统计各字段缺失数量
credit_card_missing = (
    credit_card
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


# 2. 计算缺失比例
credit_card_missing[
    "missing_rate"
] = (
    credit_card_missing[
        "missing_count"
    ]
    / len(credit_card)
)


# 3. 查看结果
credit_card_missing

In [ ]:
# 查看信用卡合同状态分布
credit_card[
    "NAME_CONTRACT_STATUS"
].value_counts(
    dropna=False
)

In [ ]:
credit_card[
    [
        "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL",
        "AMT_DRAWINGS_CURRENT",
        "AMT_DRAWINGS_ATM_CURRENT",
        "AMT_DRAWINGS_POS_CURRENT",
        "AMT_PAYMENT_CURRENT",
        "AMT_PAYMENT_TOTAL_CURRENT",
        "AMT_INST_MIN_REGULARITY",
        "AMT_TOTAL_RECEIVABLE"
    ]
].describe().T

In [ ]:
credit_card[
    [
        "SK_DPD",
        "SK_DPD_DEF"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 1】
# 建立信用卡月度使用、还款和逾期辅助特征
# ============================================================


# 1. 建立信用额度使用率
credit_card["CC_CREDIT_UTILIZATION_RATIO"] = (
    credit_card["AMT_BALANCE"]
    / credit_card["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
)


# 2. 标记当月是否存在信用卡余额
credit_card["IS_CC_BALANCE_USED"] = (
    credit_card["AMT_BALANCE"] > 0
).astype("int8")


# 3. 标记当月是否发生任何取用行为
credit_card["IS_CC_DRAWING"] = (
    credit_card["AMT_DRAWINGS_CURRENT"] > 0
).astype("int8")


# 4. 标记当月是否发生 ATM 取现
credit_card["IS_CC_ATM_DRAWING"] = (
    credit_card["AMT_DRAWINGS_ATM_CURRENT"] > 0
).astype("int8")


# 5. 标记当月是否发生 POS 消费
credit_card["IS_CC_POS_DRAWING"] = (
    credit_card["AMT_DRAWINGS_POS_CURRENT"] > 0
).astype("int8")


# 6. 计算实际还款金额与最低应还金额差值
credit_card["CC_PAYMENT_MIN_DIFF"] = (
    credit_card["AMT_PAYMENT_TOTAL_CURRENT"]
    - credit_card["AMT_INST_MIN_REGULARITY"]
)


# 7. 标记当月是否少于最低还款额
credit_card["IS_CC_UNDER_MIN_PAYMENT"] = (
    credit_card["CC_PAYMENT_MIN_DIFF"] < 0
).astype("int8")


# 8. 建立低于最低还款额的金额
credit_card["CC_UNDER_MIN_PAYMENT_AMOUNT"] = (
    -credit_card["CC_PAYMENT_MIN_DIFF"]
).clip(lower=0)


# 9. 建立最低还款完成比例
credit_card["CC_MIN_PAYMENT_RATIO"] = (
    credit_card["AMT_PAYMENT_TOTAL_CURRENT"]
    / credit_card["AMT_INST_MIN_REGULARITY"].replace(0, np.nan)
)


# 10. 标记当月是否发生原始逾期
credit_card["IS_CC_DPD"] = (
    credit_card["SK_DPD"] > 0
).astype("int8")


# 11. 标记当月是否发生容忍后逾期
credit_card["IS_CC_DPD_DEF"] = (
    credit_card["SK_DPD_DEF"] > 0
).astype("int8")


# 12. 标记当月是否发生30天及以上严重逾期
credit_card["IS_CC_SEVERE_DPD"] = (
    credit_card["SK_DPD"] >= 30
).astype("int8")


# 13. 标记当月是否处于活跃状态
credit_card["IS_CC_ACTIVE"] = (
    credit_card["NAME_CONTRACT_STATUS"] == "Active"
).astype("int8")


# 14. 标记当月是否处于完成状态
credit_card["IS_CC_COMPLETED"] = (
    credit_card["NAME_CONTRACT_STATUS"] == "Completed"
).astype("int8")


# 15. 查看结果
credit_card[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "MONTHS_BALANCE",
        "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL",
        "CC_CREDIT_UTILIZATION_RATIO",
        "AMT_DRAWINGS_CURRENT",
        "IS_CC_DRAWING",
        "IS_CC_ATM_DRAWING",
        "IS_CC_POS_DRAWING",
        "AMT_INST_MIN_REGULARITY",
        "AMT_PAYMENT_TOTAL_CURRENT",
        "CC_PAYMENT_MIN_DIFF",
        "IS_CC_UNDER_MIN_PAYMENT",
        "CC_UNDER_MIN_PAYMENT_AMOUNT",
        "CC_MIN_PAYMENT_RATIO",
        "SK_DPD",
        "SK_DPD_DEF",
        "IS_CC_DPD",
        "IS_CC_DPD_DEF",
        "IS_CC_SEVERE_DPD",
        "IS_CC_ACTIVE",
        "IS_CC_COMPLETED"
    ]
].head(10)

In [ ]:
# 1. 检查比例和金额分布
credit_card[
    [
        "CC_CREDIT_UTILIZATION_RATIO",
        "CC_PAYMENT_MIN_DIFF",
        "CC_UNDER_MIN_PAYMENT_AMOUNT",
        "CC_MIN_PAYMENT_RATIO"
    ]
].describe().T

In [ ]:
print(
    "使用信用额度的月度记录数量:",
    credit_card["IS_CC_BALANCE_USED"].sum()
)

print(
    "发生ATM取现的月度记录数量:",
    credit_card["IS_CC_ATM_DRAWING"].sum()
)

print(
    "低于最低还款额的月度记录数量:",
    credit_card["IS_CC_UNDER_MIN_PAYMENT"].sum()
)

print(
    "发生逾期的月度记录数量:",
    credit_card["IS_CC_DPD"].sum()
)

print(
    "发生30天及以上逾期的月度记录数量:",
    credit_card["IS_CC_SEVERE_DPD"].sum()
)

In [ ]:
# ============================================================
# 【特征工程 2】
# 建立信用卡账户级历史表现特征
# ============================================================


# 1. 找出每张信用卡距离当前申请日最近的一条月度记录
credit_card_latest_month = (
    credit_card
    .sort_values(
        [
            "SK_ID_PREV",
            "MONTHS_BALANCE"
        ]
    )
    .groupby("SK_ID_PREV")
    .tail(1)
    [
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
            "MONTHS_BALANCE",
            "AMT_BALANCE",
            "AMT_CREDIT_LIMIT_ACTUAL",
            "CC_CREDIT_UTILIZATION_RATIO",
            "AMT_TOTAL_RECEIVABLE",
            "NAME_CONTRACT_STATUS",
            "IS_CC_ACTIVE",
            "IS_CC_COMPLETED"
        ]
    ]
    .rename(
        columns={
            "MONTHS_BALANCE": "CC_LATEST_MONTH",
            "AMT_BALANCE": "CC_LATEST_BALANCE",
            "AMT_CREDIT_LIMIT_ACTUAL": "CC_LATEST_CREDIT_LIMIT",
            "CC_CREDIT_UTILIZATION_RATIO": "CC_LATEST_UTILIZATION_RATIO",
            "AMT_TOTAL_RECEIVABLE": "CC_LATEST_TOTAL_RECEIVABLE",
            "NAME_CONTRACT_STATUS": "CC_LATEST_CONTRACT_STATUS",
            "IS_CC_ACTIVE": "CC_LATEST_IS_ACTIVE",
            "IS_CC_COMPLETED": "CC_LATEST_IS_COMPLETED"
        }
    )
)


# 2. 按信用卡账户聚合全部历史月份表现
credit_card_account_history_features = (
    credit_card
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV"
        ]
    )
    .agg(
        # 账户历史月份数量
        CC_HISTORY_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        # 账户最早出现月份
        CC_EARLIEST_MONTH=(
            "MONTHS_BALANCE",
            "min"
        ),

        # 账户最近出现月份
        CC_MOST_RECENT_MONTH=(
            "MONTHS_BALANCE",
            "max"
        ),

        # 平均信用卡余额
        CC_AVG_BALANCE=(
            "AMT_BALANCE",
            "mean"
        ),

        # 最大信用卡余额
        CC_MAX_BALANCE=(
            "AMT_BALANCE",
            "max"
        ),

        # 平均信用额度
        CC_AVG_CREDIT_LIMIT=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "mean"
        ),

        # 最大信用额度
        CC_MAX_CREDIT_LIMIT=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "max"
        ),

        # 平均额度使用率
        CC_AVG_UTILIZATION_RATIO=(
            "CC_CREDIT_UTILIZATION_RATIO",
            "mean"
        ),

        # 最大额度使用率
        CC_MAX_UTILIZATION_RATIO=(
            "CC_CREDIT_UTILIZATION_RATIO",
            "max"
        ),

        # 有余额月份数量
        CC_BALANCE_USED_MONTH_COUNT=(
            "IS_CC_BALANCE_USED",
            "sum"
        ),

        # 发生取用行为的月份数量
        CC_DRAWING_MONTH_COUNT=(
            "IS_CC_DRAWING",
            "sum"
        ),

        # ATM取现月份数量
        CC_ATM_DRAWING_MONTH_COUNT=(
            "IS_CC_ATM_DRAWING",
            "sum"
        ),

        # POS消费月份数量
        CC_POS_DRAWING_MONTH_COUNT=(
            "IS_CC_POS_DRAWING",
            "sum"
        ),

        # 累计总取用金额
        CC_TOTAL_DRAWINGS_AMOUNT=(
            "AMT_DRAWINGS_CURRENT",
            "sum"
        ),

        # 累计ATM取现金额
        CC_TOTAL_ATM_DRAWINGS_AMOUNT=(
            "AMT_DRAWINGS_ATM_CURRENT",
            "sum"
        ),

        # 累计POS消费金额
        CC_TOTAL_POS_DRAWINGS_AMOUNT=(
            "AMT_DRAWINGS_POS_CURRENT",
            "sum"
        ),

        # 平均每月总还款金额
        CC_AVG_PAYMENT_TOTAL=(
            "AMT_PAYMENT_TOTAL_CURRENT",
            "mean"
        ),

        # 累计总还款金额
        CC_TOTAL_PAYMENT_AMOUNT=(
            "AMT_PAYMENT_TOTAL_CURRENT",
            "sum"
        ),

        # 平均最低应还金额
        CC_AVG_MIN_PAYMENT=(
            "AMT_INST_MIN_REGULARITY",
            "mean"
        ),

        # 低于最低还款额的月份数量
        CC_UNDER_MIN_PAYMENT_MONTH_COUNT=(
            "IS_CC_UNDER_MIN_PAYMENT",
            "sum"
        ),

        # 累计低于最低还款额的金额
        CC_TOTAL_UNDER_MIN_PAYMENT_AMOUNT=(
            "CC_UNDER_MIN_PAYMENT_AMOUNT",
            "sum"
        ),

        # 平均最低还款完成比例
        CC_AVG_MIN_PAYMENT_RATIO=(
            "CC_MIN_PAYMENT_RATIO",
            "mean"
        ),

        # 最低的最低还款完成比例
        CC_MIN_MIN_PAYMENT_RATIO=(
            "CC_MIN_PAYMENT_RATIO",
            "min"
        ),

        # 原始逾期月份数量
        CC_DPD_MONTH_COUNT=(
            "IS_CC_DPD",
            "sum"
        ),

        # 容忍后逾期月份数量
        CC_DPD_DEF_MONTH_COUNT=(
            "IS_CC_DPD_DEF",
            "sum"
        ),

        # 严重逾期月份数量
        CC_SEVERE_DPD_MONTH_COUNT=(
            "IS_CC_SEVERE_DPD",
            "sum"
        ),

        # 平均逾期天数
        CC_AVG_DPD=(
            "SK_DPD",
            "mean"
        ),

        # 最大逾期天数
        CC_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        # 平均容忍后逾期天数
        CC_AVG_DPD_DEF=(
            "SK_DPD_DEF",
            "mean"
        ),

        # 最大容忍后逾期天数
        CC_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        )
    )
    .reset_index()
)


# 3. 建立账户级有余额月份比例
credit_card_account_history_features[
    "CC_BALANCE_USED_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_BALANCE_USED_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 4. 建立账户级取用月份比例
credit_card_account_history_features[
    "CC_DRAWING_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_DRAWING_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立账户级ATM取现月份比例
credit_card_account_history_features[
    "CC_ATM_DRAWING_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_ATM_DRAWING_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 6. 建立账户级低于最低还款月份比例
credit_card_account_history_features[
    "CC_UNDER_MIN_PAYMENT_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_UNDER_MIN_PAYMENT_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 7. 建立账户级逾期月份比例
credit_card_account_history_features[
    "CC_DPD_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_DPD_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 8. 建立账户级容忍后逾期月份比例
credit_card_account_history_features[
    "CC_DPD_DEF_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_DPD_DEF_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 9. 建立账户级严重逾期月份比例
credit_card_account_history_features[
    "CC_SEVERE_DPD_MONTH_RATE"
] = (
    credit_card_account_history_features[
        "CC_SEVERE_DPD_MONTH_COUNT"
    ]
    / credit_card_account_history_features[
        "CC_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 10. 合并每张卡最近一个月的状态
credit_card_account_features = (
    credit_card_account_history_features
    .merge(
        credit_card_latest_month,
        on=[
            "SK_ID_CURR",
            "SK_ID_PREV"
        ],
        how="left",
        validate="one_to_one"
    )
)


# 11. 建立信用卡账户历史跨度
credit_card_account_features[
    "CC_HISTORY_SPAN_MONTHS"
] = (
    credit_card_account_features[
        "CC_MOST_RECENT_MONTH"
    ]
    - credit_card_account_features[
        "CC_EARLIEST_MONTH"
    ]
)


# 12. 查看账户级结果
credit_card_account_features.head()

In [ ]:
# 1. 查看账户级数据规模
print(
    "credit_card_account_features shape:",
    credit_card_account_features.shape
)


# 2. 查看唯一信用卡账户数量
print(
    "唯一信用卡账户数量:",
    credit_card_account_features[
        "SK_ID_PREV"
    ].nunique()
)


# 3. 检查重复信用卡账户数量
print(
    "重复信用卡账户数量:",
    credit_card_account_features[
        "SK_ID_PREV"
    ].duplicated().sum()
)

In [ ]:
credit_card_account_features[
    [
        "CC_BALANCE_USED_MONTH_RATE",
        "CC_DRAWING_MONTH_RATE",
        "CC_ATM_DRAWING_MONTH_RATE",
        "CC_UNDER_MIN_PAYMENT_MONTH_RATE",
        "CC_DPD_MONTH_RATE",
        "CC_DPD_DEF_MONTH_RATE",
        "CC_SEVERE_DPD_MONTH_RATE"
    ]
].describe().T

In [ ]:
credit_card_account_features[
    [
        "CC_HISTORY_MONTH_COUNT",
        "CC_HISTORY_SPAN_MONTHS",
        "CC_AVG_BALANCE",
        "CC_MAX_BALANCE",
        "CC_TOTAL_DRAWINGS_AMOUNT",
        "CC_TOTAL_PAYMENT_AMOUNT",
        "CC_TOTAL_UNDER_MIN_PAYMENT_AMOUNT",
        "CC_AVG_DPD",
        "CC_MAX_DPD"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 3】
# 建立客户级信用卡历史表现特征
# ============================================================


# 1. 按客户聚合信用卡账户表现
credit_card_customer_features = (
    credit_card_account_features
    .groupby("SK_ID_CURR")
    .agg(
        # 客户历史信用卡账户数量
        CC_ACCOUNT_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        # 客户累计信用卡历史月份数量
        CC_TOTAL_HISTORY_MONTH_COUNT=(
            "CC_HISTORY_MONTH_COUNT",
            "sum"
        ),

        # 客户单张卡平均历史月份数量
        CC_AVG_HISTORY_MONTH_COUNT=(
            "CC_HISTORY_MONTH_COUNT",
            "mean"
        ),

        # 客户最长信用卡历史月份数量
        CC_MAX_HISTORY_MONTH_COUNT=(
            "CC_HISTORY_MONTH_COUNT",
            "max"
        ),

        # 客户最早信用卡记录月份
        CC_CUSTOMER_EARLIEST_MONTH=(
            "CC_EARLIEST_MONTH",
            "min"
        ),

        # 客户最近信用卡记录月份
        CC_CUSTOMER_MOST_RECENT_MONTH=(
            "CC_MOST_RECENT_MONTH",
            "max"
        ),

        # 客户多张卡平均余额
        CC_AVG_ACCOUNT_BALANCE=(
            "CC_AVG_BALANCE",
            "mean"
        ),

        # 客户历史最高余额
        CC_MAX_BALANCE=(
            "CC_MAX_BALANCE",
            "max"
        ),

        # 客户多张卡平均信用额度
        CC_AVG_ACCOUNT_CREDIT_LIMIT=(
            "CC_AVG_CREDIT_LIMIT",
            "mean"
        ),

        # 客户历史最高信用额度
        CC_MAX_CREDIT_LIMIT=(
            "CC_MAX_CREDIT_LIMIT",
            "max"
        ),

        # 客户多张卡平均额度使用率
        CC_AVG_ACCOUNT_UTILIZATION_RATIO=(
            "CC_AVG_UTILIZATION_RATIO",
            "mean"
        ),

        # 客户历史最高额度使用率
        CC_MAX_UTILIZATION_RATIO=(
            "CC_MAX_UTILIZATION_RATIO",
            "max"
        ),

        # 客户最近余额合计
        CC_TOTAL_LATEST_BALANCE=(
            "CC_LATEST_BALANCE",
            "sum"
        ),

        # 客户最近信用额度合计
        CC_TOTAL_LATEST_CREDIT_LIMIT=(
            "CC_LATEST_CREDIT_LIMIT",
            "sum"
        ),

        # 客户最近总应收金额合计
        CC_TOTAL_LATEST_RECEIVABLE=(
            "CC_LATEST_TOTAL_RECEIVABLE",
            "sum"
        ),

        # 客户累计有余额月份数量
        CC_TOTAL_BALANCE_USED_MONTH_COUNT=(
            "CC_BALANCE_USED_MONTH_COUNT",
            "sum"
        ),

        # 客户累计发生取用行为的月份数量
        CC_TOTAL_DRAWING_MONTH_COUNT=(
            "CC_DRAWING_MONTH_COUNT",
            "sum"
        ),

        # 客户累计ATM取现月份数量
        CC_TOTAL_ATM_DRAWING_MONTH_COUNT=(
            "CC_ATM_DRAWING_MONTH_COUNT",
            "sum"
        ),

        # 客户累计POS消费月份数量
        CC_TOTAL_POS_DRAWING_MONTH_COUNT=(
            "CC_POS_DRAWING_MONTH_COUNT",
            "sum"
        ),

        # 客户累计总取用金额
        CC_TOTAL_DRAWINGS_AMOUNT=(
            "CC_TOTAL_DRAWINGS_AMOUNT",
            "sum"
        ),

        # 客户累计ATM取现金额
        CC_TOTAL_ATM_DRAWINGS_AMOUNT=(
            "CC_TOTAL_ATM_DRAWINGS_AMOUNT",
            "sum"
        ),

        # 客户累计POS消费金额
        CC_TOTAL_POS_DRAWINGS_AMOUNT=(
            "CC_TOTAL_POS_DRAWINGS_AMOUNT",
            "sum"
        ),

        # 客户累计总还款金额
        CC_TOTAL_PAYMENT_AMOUNT=(
            "CC_TOTAL_PAYMENT_AMOUNT",
            "sum"
        ),

        # 客户累计低于最低还款额月份数量
        CC_TOTAL_UNDER_MIN_PAYMENT_MONTH_COUNT=(
            "CC_UNDER_MIN_PAYMENT_MONTH_COUNT",
            "sum"
        ),

        # 客户累计低于最低还款额金额
        CC_TOTAL_UNDER_MIN_PAYMENT_AMOUNT=(
            "CC_TOTAL_UNDER_MIN_PAYMENT_AMOUNT",
            "sum"
        ),

        # 客户多张卡平均最低还款完成比例
        CC_AVG_ACCOUNT_MIN_PAYMENT_RATIO=(
            "CC_AVG_MIN_PAYMENT_RATIO",
            "mean"
        ),

        # 客户历史最低还款完成比例
        CC_MIN_PAYMENT_RATIO=(
            "CC_MIN_MIN_PAYMENT_RATIO",
            "min"
        ),

        # 客户累计原始逾期月份数量
        CC_TOTAL_DPD_MONTH_COUNT=(
            "CC_DPD_MONTH_COUNT",
            "sum"
        ),

        # 客户累计容忍后逾期月份数量
        CC_TOTAL_DPD_DEF_MONTH_COUNT=(
            "CC_DPD_DEF_MONTH_COUNT",
            "sum"
        ),

        # 客户累计严重逾期月份数量
        CC_TOTAL_SEVERE_DPD_MONTH_COUNT=(
            "CC_SEVERE_DPD_MONTH_COUNT",
            "sum"
        ),

        # 客户多张卡平均逾期天数
        CC_AVG_ACCOUNT_DPD=(
            "CC_AVG_DPD",
            "mean"
        ),

        # 客户历史最大逾期天数
        CC_MAX_DPD=(
            "CC_MAX_DPD",
            "max"
        ),

        # 客户多张卡平均容忍后逾期天数
        CC_AVG_ACCOUNT_DPD_DEF=(
            "CC_AVG_DPD_DEF",
            "mean"
        ),

        # 客户历史最大容忍后逾期天数
        CC_MAX_DPD_DEF=(
            "CC_MAX_DPD_DEF",
            "max"
        ),

        # 最近仍处于活跃状态的信用卡账户数量
        CC_ACTIVE_ACCOUNT_COUNT=(
            "CC_LATEST_IS_ACTIVE",
            "sum"
        ),

        # 最近已完成的信用卡账户数量
        CC_COMPLETED_ACCOUNT_COUNT=(
            "CC_LATEST_IS_COMPLETED",
            "sum"
        )
    )
    .reset_index()
)


# 2. 建立客户整体有余额月份比例
credit_card_customer_features[
    "CC_TOTAL_BALANCE_USED_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_BALANCE_USED_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 3. 建立客户整体取用月份比例
credit_card_customer_features[
    "CC_TOTAL_DRAWING_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_DRAWING_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 4. 建立客户整体ATM取现月份比例
credit_card_customer_features[
    "CC_TOTAL_ATM_DRAWING_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_ATM_DRAWING_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立客户整体低于最低还款月份比例
credit_card_customer_features[
    "CC_TOTAL_UNDER_MIN_PAYMENT_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_UNDER_MIN_PAYMENT_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 6. 建立客户整体逾期月份比例
credit_card_customer_features[
    "CC_TOTAL_DPD_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_DPD_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 7. 建立客户整体容忍后逾期月份比例
credit_card_customer_features[
    "CC_TOTAL_DPD_DEF_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_DPD_DEF_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 8. 建立客户整体严重逾期月份比例
credit_card_customer_features[
    "CC_TOTAL_SEVERE_DPD_MONTH_RATE"
] = (
    credit_card_customer_features[
        "CC_TOTAL_SEVERE_DPD_MONTH_COUNT"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 9. 建立客户最近整体额度使用率
credit_card_customer_features[
    "CC_TOTAL_LATEST_UTILIZATION_RATIO"
] = (
    credit_card_customer_features[
        "CC_TOTAL_LATEST_BALANCE"
    ]
    / credit_card_customer_features[
        "CC_TOTAL_LATEST_CREDIT_LIMIT"
    ].replace(0, np.nan)
)


# 10. 建立活跃信用卡账户比例
credit_card_customer_features[
    "CC_ACTIVE_ACCOUNT_RATE"
] = (
    credit_card_customer_features[
        "CC_ACTIVE_ACCOUNT_COUNT"
    ]
    / credit_card_customer_features[
        "CC_ACCOUNT_COUNT"
    ].replace(0, np.nan)
)


# 11. 建立客户信用卡历史跨度
credit_card_customer_features[
    "CC_CUSTOMER_HISTORY_SPAN_MONTHS"
] = (
    credit_card_customer_features[
        "CC_CUSTOMER_MOST_RECENT_MONTH"
    ]
    - credit_card_customer_features[
        "CC_CUSTOMER_EARLIEST_MONTH"
    ]
)


# 12. 查看客户级结果
credit_card_customer_features.head()

In [ ]:
# 1. 查看客户级特征表规模
print(
    "credit_card_customer_features shape:",
    credit_card_customer_features.shape
)


# 2. 查看唯一客户数量
print(
    "唯一客户数量:",
    credit_card_customer_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    credit_card_customer_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
credit_card_customer_features[
    [
        "CC_TOTAL_BALANCE_USED_MONTH_RATE",
        "CC_TOTAL_DRAWING_MONTH_RATE",
        "CC_TOTAL_ATM_DRAWING_MONTH_RATE",
        "CC_TOTAL_UNDER_MIN_PAYMENT_MONTH_RATE",
        "CC_TOTAL_DPD_MONTH_RATE",
        "CC_TOTAL_DPD_DEF_MONTH_RATE",
        "CC_TOTAL_SEVERE_DPD_MONTH_RATE",
        "CC_ACTIVE_ACCOUNT_RATE"
    ]
].describe().T

In [ ]:
credit_card_customer_features[
    [
        "CC_ACCOUNT_COUNT",
        "CC_TOTAL_HISTORY_MONTH_COUNT",
        "CC_TOTAL_LATEST_BALANCE",
        "CC_TOTAL_LATEST_CREDIT_LIMIT",
        "CC_TOTAL_DRAWINGS_AMOUNT",
        "CC_TOTAL_PAYMENT_AMOUNT",
        "CC_TOTAL_UNDER_MIN_PAYMENT_AMOUNT",
        "CC_TOTAL_DPD_MONTH_COUNT",
        "CC_TOTAL_SEVERE_DPD_MONTH_COUNT",
        "CC_MAX_DPD",
        "CC_CUSTOMER_HISTORY_SPAN_MONTHS"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 4】
# 建立客户最近一年信用卡行为特征
# ============================================================


# 1. 筛选申请日前最近12个月的信用卡记录
recent_credit_card = (
    credit_card[
        credit_card["MONTHS_BALANCE"] >= -12
    ]
    .copy()
)


# 2. 查看最近一年数据规模
print(
    "最近一年信用卡月度记录数量:",
    recent_credit_card.shape[0]
)

print(
    "最近一年涉及客户数量:",
    recent_credit_card["SK_ID_CURR"].nunique()
)


# 3. 按客户聚合最近一年信用卡表现
credit_card_recent_features = (
    recent_credit_card
    .groupby("SK_ID_CURR")
    .agg(
        # 最近一年信用卡月度记录数量
        CC_RECENT_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        # 最近一年涉及的信用卡账户数量
        CC_RECENT_ACCOUNT_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        # 最近一年平均信用卡余额
        CC_RECENT_AVG_BALANCE=(
            "AMT_BALANCE",
            "mean"
        ),

        # 最近一年最高信用卡余额
        CC_RECENT_MAX_BALANCE=(
            "AMT_BALANCE",
            "max"
        ),

        # 最近一年平均信用额度
        CC_RECENT_AVG_CREDIT_LIMIT=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "mean"
        ),

        # 最近一年平均额度使用率
        CC_RECENT_AVG_UTILIZATION_RATIO=(
            "CC_CREDIT_UTILIZATION_RATIO",
            "mean"
        ),

        # 最近一年最高额度使用率
        CC_RECENT_MAX_UTILIZATION_RATIO=(
            "CC_CREDIT_UTILIZATION_RATIO",
            "max"
        ),

        # 最近一年有余额月份数量
        CC_RECENT_BALANCE_USED_MONTH_COUNT=(
            "IS_CC_BALANCE_USED",
            "sum"
        ),

        # 最近一年发生取用行为的月份数量
        CC_RECENT_DRAWING_MONTH_COUNT=(
            "IS_CC_DRAWING",
            "sum"
        ),

        # 最近一年ATM取现月份数量
        CC_RECENT_ATM_DRAWING_MONTH_COUNT=(
            "IS_CC_ATM_DRAWING",
            "sum"
        ),

        # 最近一年POS消费月份数量
        CC_RECENT_POS_DRAWING_MONTH_COUNT=(
            "IS_CC_POS_DRAWING",
            "sum"
        ),

        # 最近一年累计总取用金额
        CC_RECENT_TOTAL_DRAWINGS_AMOUNT=(
            "AMT_DRAWINGS_CURRENT",
            "sum"
        ),

        # 最近一年累计ATM取现金额
        CC_RECENT_TOTAL_ATM_DRAWINGS_AMOUNT=(
            "AMT_DRAWINGS_ATM_CURRENT",
            "sum"
        ),

        # 最近一年累计POS消费金额
        CC_RECENT_TOTAL_POS_DRAWINGS_AMOUNT=(
            "AMT_DRAWINGS_POS_CURRENT",
            "sum"
        ),

        # 最近一年累计总还款金额
        CC_RECENT_TOTAL_PAYMENT_AMOUNT=(
            "AMT_PAYMENT_TOTAL_CURRENT",
            "sum"
        ),

        # 最近一年平均最低应还金额
        CC_RECENT_AVG_MIN_PAYMENT=(
            "AMT_INST_MIN_REGULARITY",
            "mean"
        ),

        # 最近一年低于最低还款额的月份数量
        CC_RECENT_UNDER_MIN_PAYMENT_MONTH_COUNT=(
            "IS_CC_UNDER_MIN_PAYMENT",
            "sum"
        ),

        # 最近一年累计低于最低还款额金额
        CC_RECENT_TOTAL_UNDER_MIN_PAYMENT_AMOUNT=(
            "CC_UNDER_MIN_PAYMENT_AMOUNT",
            "sum"
        ),

        # 最近一年平均最低还款完成比例
        CC_RECENT_AVG_MIN_PAYMENT_RATIO=(
            "CC_MIN_PAYMENT_RATIO",
            "mean"
        ),

        # 最近一年最低的最低还款完成比例
        CC_RECENT_MIN_PAYMENT_RATIO=(
            "CC_MIN_PAYMENT_RATIO",
            "min"
        ),

        # 最近一年原始逾期月份数量
        CC_RECENT_DPD_MONTH_COUNT=(
            "IS_CC_DPD",
            "sum"
        ),

        # 最近一年容忍后逾期月份数量
        CC_RECENT_DPD_DEF_MONTH_COUNT=(
            "IS_CC_DPD_DEF",
            "sum"
        ),

        # 最近一年严重逾期月份数量
        CC_RECENT_SEVERE_DPD_MONTH_COUNT=(
            "IS_CC_SEVERE_DPD",
            "sum"
        ),

        # 最近一年平均原始逾期天数
        CC_RECENT_AVG_DPD=(
            "SK_DPD",
            "mean"
        ),

        # 最近一年最大原始逾期天数
        CC_RECENT_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        # 最近一年平均容忍后逾期天数
        CC_RECENT_AVG_DPD_DEF=(
            "SK_DPD_DEF",
            "mean"
        ),

        # 最近一年最大容忍后逾期天数
        CC_RECENT_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        )
    )
    .reset_index()
)


# 4. 建立最近一年有余额月份比例
credit_card_recent_features[
    "CC_RECENT_BALANCE_USED_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_BALANCE_USED_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立最近一年取用月份比例
credit_card_recent_features[
    "CC_RECENT_DRAWING_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_DRAWING_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 6. 建立最近一年ATM取现月份比例
credit_card_recent_features[
    "CC_RECENT_ATM_DRAWING_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_ATM_DRAWING_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 7. 建立最近一年低于最低还款月份比例
credit_card_recent_features[
    "CC_RECENT_UNDER_MIN_PAYMENT_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_UNDER_MIN_PAYMENT_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 8. 建立最近一年逾期月份比例
credit_card_recent_features[
    "CC_RECENT_DPD_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_DPD_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 9. 建立最近一年容忍后逾期月份比例
credit_card_recent_features[
    "CC_RECENT_DPD_DEF_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_DPD_DEF_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 10. 建立最近一年严重逾期月份比例
credit_card_recent_features[
    "CC_RECENT_SEVERE_DPD_MONTH_RATE"
] = (
    credit_card_recent_features[
        "CC_RECENT_SEVERE_DPD_MONTH_COUNT"
    ]
    / credit_card_recent_features[
        "CC_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 11. 查看客户最近一年信用卡特征
credit_card_recent_features.head()

In [ ]:
# 1. 查看近期客户级特征规模
print(
    "credit_card_recent_features shape:",
    credit_card_recent_features.shape
)


# 2. 查看唯一客户数量
print(
    "唯一客户数量:",
    credit_card_recent_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    credit_card_recent_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
credit_card_recent_features[
    [
        "CC_RECENT_BALANCE_USED_MONTH_RATE",
        "CC_RECENT_DRAWING_MONTH_RATE",
        "CC_RECENT_ATM_DRAWING_MONTH_RATE",
        "CC_RECENT_UNDER_MIN_PAYMENT_MONTH_RATE",
        "CC_RECENT_DPD_MONTH_RATE",
        "CC_RECENT_DPD_DEF_MONTH_RATE",
        "CC_RECENT_SEVERE_DPD_MONTH_RATE"
    ]
].describe().T

In [ ]:
credit_card_recent_features[
    [
        "CC_RECENT_MONTH_COUNT",
        "CC_RECENT_ACCOUNT_COUNT",
        "CC_RECENT_AVG_BALANCE",
        "CC_RECENT_MAX_BALANCE",
        "CC_RECENT_TOTAL_DRAWINGS_AMOUNT",
        "CC_RECENT_TOTAL_PAYMENT_AMOUNT",
        "CC_RECENT_TOTAL_UNDER_MIN_PAYMENT_AMOUNT",
        "CC_RECENT_AVG_DPD",
        "CC_RECENT_MAX_DPD"
    ]
].describe().T

In [ ]:
# ============================================================
# 【合并步骤】
# 合并信用卡客户级整体特征与近期特征
# ============================================================


# 1. 以客户整体历史特征作为主表
credit_card_features = (
    credit_card_customer_features

    # 2. 合并客户最近一年信用卡特征
    .merge(
        credit_card_recent_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)


# 3. 查看最终结果
credit_card_features.head()

In [ ]:
# 1. 查看最终数据规模
print(
    "credit_card_features shape:",
    credit_card_features.shape
)


# 2. 查看唯一客户数量
print(
    "唯一客户数量:",
    credit_card_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    credit_card_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)


# 4. 与原始表唯一客户数量对比
print(
    "原始信用卡唯一客户数量:",
    credit_card[
        "SK_ID_CURR"
    ].nunique()
)

In [ ]:
duplicate_suffix_columns = [
    col
    for col in credit_card_features.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(
    "重复后缀字段:",
    duplicate_suffix_columns
)

In [ ]:
# 1. 统计缺失数量
credit_card_missing_summary = (
    credit_card_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


# 2. 计算缺失比例
credit_card_missing_summary[
    "missing_rate"
] = (
    credit_card_missing_summary[
        "missing_count"
    ]
    / len(credit_card_features)
)


# 3. 查看缺失最多的字段
credit_card_missing_summary.head(20)

In [ ]:
# 1. 找出所有比例字段
credit_card_ratio_columns = [
    col
    for col in credit_card_features.columns
    if "RATE" in col or "RATIO" in col
]


# 2. 查看比例字段分布
credit_card_features[
    credit_card_ratio_columns
].describe().T

In [ ]:
key_credit_card_features = [
    "CC_ACCOUNT_COUNT",
    "CC_TOTAL_HISTORY_MONTH_COUNT",
    "CC_TOTAL_LATEST_BALANCE",
    "CC_TOTAL_LATEST_CREDIT_LIMIT",
    "CC_TOTAL_LATEST_UTILIZATION_RATIO",
    "CC_TOTAL_DRAWINGS_AMOUNT",
    "CC_TOTAL_ATM_DRAWINGS_AMOUNT",
    "CC_TOTAL_PAYMENT_AMOUNT",
    "CC_TOTAL_UNDER_MIN_PAYMENT_MONTH_COUNT",
    "CC_TOTAL_UNDER_MIN_PAYMENT_MONTH_RATE",
    "CC_TOTAL_DPD_MONTH_COUNT",
    "CC_TOTAL_DPD_MONTH_RATE",
    "CC_MAX_DPD",
    "CC_RECENT_MONTH_COUNT",
    "CC_RECENT_AVG_UTILIZATION_RATIO",
    "CC_RECENT_UNDER_MIN_PAYMENT_MONTH_RATE",
    "CC_RECENT_DPD_MONTH_RATE",
    "CC_RECENT_MAX_DPD"
]


for col in key_credit_card_features:
    print(
        col,
        col in credit_card_features.columns
    )

In [ ]:
from pathlib import Path


# 1. 建立输出路径
output_path = Path(
    "data/processed/credit_card_features.parquet"
)


# 2. 保存最终客户级特征表
credit_card_features.to_parquet(
    output_path,
    index=False
)


# 3. 检查文件是否保存成功
print(
    "文件是否存在:",
    output_path.exists()
)

print(
    "保存位置:",
    output_path.resolve()
)

print(
    "最终 Shape:",
    credit_card_features.shape
)